In [ ]:
# Import necessary libraries
from pathlib import Path
import re, json
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase

In [ ]:
# Source file
input_path = Path(r"data generation\conversation generation\raw conversations\final_few_shot_raw_conversations_gpt4.txt")

# Make a dictionary for the evaluation dialogues 
output_dir = Path(r"data evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "evaluation_dialogues.jsonl"
output_path

In [ ]:
# Read dialogues 
text = input_path.read_text(encoding="utf-8", errors="ignore")

# Normalize newlines
text = text.replace("\r\n", "\n").replace("\r", "\n")

# Split conversations on one-or-more blank lines
blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]

# Convert "A:/P:" to "Agent:/Patient:" to reduce ambiguity to the LLM
def to_standard_dialogue(block: str) -> str:
    lines = []
    for line in block.split("\n"):
        if line.startswith("A:"):
            lines.append("Agent:" + line[2:])
        elif line.startswith("P:"):
            lines.append("Patient:" + line[2:])
        else:
            if line.strip():
                lines.append(line.strip())
    return "\n".join(lines).strip()

# Write JSONL
with output_path.open("w", encoding="utf-8") as f:
    for i, block in enumerate(blocks, 1):
        dlg = to_standard_dialogue(block)
        item = {"dialogue_id": f"d{i:04d}", "dialogue": dlg}
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

len(blocks), output_path

In [6]:
df_preview = pd.read_json(output_path, lines=True).head(10)
df_preview

,dialogue_id,dialogue
0,d0001,"Agent: Hey Jan, I see that you're a fan of cyc..."
1,d0002,Agent: I noticed that you've been experiencing...
2,d0003,Patient: I've noticed some blurry vision latel...
3,d0004,Patient: I've been more thirsty than usual the...
4,d0005,"Agent: Jan, I see that you're doing a great jo..."
5,d0006,Patient: I've been having a difficult time man...
6,d0007,Patient: I've been noticing tingling in my fee...
7,d0008,"Agent: Jan, I understand that you've been mana..."
8,d0009,Patient: I've been experiencing tingling in my...
9,d0010,"Agent: Jan, have you been experiencing any cha..."


In [ ]:
# Check if the openai api key is loaded
print("Key loaded:", "OPENAI_API_KEY" in os.environ)

Key loaded: True


In [ ]:
# Initialize OpenAI client
client = OpenAI()

# File paths
inp_path  = Path(r"data evaluation\evaluation_dialogues.jsonl")
long_out = Path(r"data evaluation\deepeval_scores.long.jsonl")
wide_out = Path(r"data evaluation\deepeval_scores.wide.csv")

# Model config
MODEL = os.environ.get("GEVAL_MODEL", "gpt-4o")
TEMPERATURE = 0.0 # deterministic; consistency across calls

# Metrics and rubrics
METRICS = ["fluency", "naturalness", "clarity", "structure"]
RUBRICS = {
    "fluency": "1: Frequent grammatical errors; hard to parse.\n2: Several errors; meaning sometimes unclear.\n3: Minor errors; overall understandable.\n4: Grammatically correct; rare minor issues.\n5: Polished, error-free, smooth reading.",
    "naturalness": "1: Stilted/robotic; unnatural phrasing.\n2: Awkward or templated; limited variation.\n3: Generally natural with a few stiff spots.\n4: Conversational and varied; flows well.\n5: Conversation sounds completely natural and realistic.",
    "clarity": "1: Confusing; hard to follow.\n2: Some confusion; imprecise wording.\n3: Understandable; could be clearer or more precise.\n4: Clear and easy to follow.\n5: Very clear; concise phrasing that aids understanding.",
    "structure": "1: Disorganized; no clear structure.\n2: Poor organization; confusing structure.\n3: Some structure; could flow better.\n4: Well-organized with logical flow or steps.\n5: Excellent structure; clear sections/steps that aid comprehension.",
}

SYSTEM_PROMPT = """
You are a careful evaluator of synthetic patient–agent conversations about Type 2 Diabetes management.  
Your task is to assess the entire conversation on one specified metric, using the provided rubric with a 1–5 scale.  

Requirements: 
- Base your judgment ONLY on the given metric and rubric.
- Evaluate the full dialogue (patient and agent turns). 
- Output must be a single valid JSON object in the specified schema. 
- DO NOT include any text outside the JSON.
- Provide ONLY a brief justification (1–2 sentences).
- Never expose or describe your internal reasoning steps.
"""

USER_TEMPLATE = """Metric: {METRIC}

Rubric (1–5):
{RUBRIC}

Instructions:
- Consider the dialogue as a whole (both patient and agent turns).
- Judge the overall quality for the specified metric.
- Output ONLY this JSON:
{{
  "probabilities": {{"1": p1, "2": p2, "3": p3, "4": p4, "5": p5}},
  "brief_reasons": "1–2 sentence justification tied to the rubric."
}}
Rules for probabilities:
- Non-negative
- Sum to 1 (±1e-6 is fine)
- Use decimals (e.g., 0.12), not percentages.

Dialogue:
{DIALOGUE}
"""



In [ ]:
def prob_to_expected(prob_dict: dict) -> float:
    """
    Convert a probability distribution to an expected value.

    """
    ks = np.arange(1, 6, dtype=float) # possible scores; 1-5
    ps = np.array([float(prob_dict.get(str(k), 0.0)) for k in range(1,6)], dtype=float)
    s = ps.sum()
    if s <= 0:
        ps = np.ones(5, dtype=float) / 5.0
    else:
        ps = ps / s
    return float((ks * ps).sum()) # compute expected value (weighted average)

def entropy(prob_dict: dict) -> float:
    """
    Calculate the entropy of a probability distribution.
    
    """
    ps = np.array([float(prob_dict.get(str(k), 0.0)) for k in range(1,6)], dtype=float)
    s = ps.sum()
    ps = (ps/s) if s > 0 else np.ones(5)/5.0
    ps = np.clip(ps, 1e-12, 1.0)
    return float(-(ps * np.log(ps)).sum())


In [ ]:
# Load dialogues
items = []
with open(inp_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            items.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipping invalid line:", line[:80], "...")
print(f"Loaded {len(items)} dialogues")


# Evaluation loop 
rows_long = []

for rec in tqdm(items, desc="Scoring"):
    did = rec["dialogue_id"]
    dialogue = rec["dialogue"]

    for metric_name in METRICS:
        user_prompt = USER_TEMPLATE.format(
            METRIC=metric_name.upper(),
            RUBRIC=RUBRICS[metric_name],
            DIALOGUE=dialogue
        )

        resp = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        content = resp.choices[0].message.content

        try:
            data = json.loads(content)
        except Exception:
            start, end = content.find("{"), content.rfind("}")
            data = json.loads(content[start:end+1]) if start != -1 and end != -1 else {"probabilities": {}}

        probs = data.get("probabilities", {})
        brief = data.get("brief_reasons", "")
        exp = prob_to_expected(probs)
        ent = entropy(probs)

        rows_long.append({
            "dialogue_id": did,
            "dimension": metric_name,
            "expected": round(exp, 4),
            "entropy": round(ent, 6),
            "probabilities": json.dumps({str(k): float(probs.get(str(k), 0.0)) for k in range(1, 6)}),
            "brief_reasons": brief,
            "raw_json": content
        })

# Save long JSONL
with open(long_out, "w", encoding="utf-8") as f:
    for row in rows_long:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Save wide CSV 
df = pd.DataFrame(rows_long)
wide = df.pivot_table(index="dialogue_id", columns="dimension", values="expected", aggfunc="first")
wide.columns = [f"{c}_score" for c in wide.columns]
ent = df.pivot_table(index="dialogue_id", columns="dimension", values="entropy", aggfunc="first")
ent.columns = [f"{c}_entropy" for c in ent.columns]
wide = wide.join(ent, how="left").reset_index()
wide.to_csv(wide_out, index=False)

print("Saved:", long_out, "and", wide_out)

